# Day 4

## Tokenizing with code

In [3]:
import tiktoken

encoding = tiktoken.encoding_for_model("gpt-4.1-mini")

tokens = encoding.encode("Hi my name is Navneet and I like white sauce paste")

In [5]:
tokens

[12194, 922, 1308, 382, 17271, 141755, 326, 357, 1299, 6461, 24524, 32307]

In [6]:
for token_id in tokens:
    token_text = encoding.decode([token_id])
    print(f"{token_id} = {token_text}")

12194 = Hi
922 =  my
1308 =  name
382 =  is
17271 =  Nav
141755 = neet
326 =  and
357 =  I
1299 =  like
6461 =  white
24524 =  sauce
32307 =  paste


In [8]:
encoding.decode([326])

' and'

# And another topic!

### The Illusion of "memory"

Many of you will know this already. But for those that don't -- this might be an "AHA" moment!

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

### You should be very comfortable with what the next cell is doing!

_I'm creating a new instance of the OpenAI Python Client library, a lightweight wrapper around making HTTP calls to an endpoint for calling the GPT LLM, or other LLM providers_

In [16]:
from openai import OpenAI

OLLAMA_BASE_URL = "http://localhost:11434/v1"

# api_key is required by the client but ignored by Ollama — any non-empty string works.
openai = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

MODEL = "minimax-m3:cloud"

### A message to OpenAI is a list of dicts

In [14]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm Navi!"}
    ]

In [19]:
response = openai.chat.completions.create(model=MODEL, messages=messages)
response.choices[0].message.content

"Hi Navi! 👋 Welcome! Nice to meet you. I'm MiniMax-M3, your AI assistant. \n\nHow can I help you today? Whether you have questions, need assistance with something, or just want to chat, I'm here for you! 😊"

### OK let's now ask a follow-up question

In [20]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What's my name?"}
    ]

In [21]:
response = openai.chat.completions.create(model=MODEL, messages=messages)
response.choices[0].message.content

"I don't have access to personal information about you, including your name. I'm an AI assistant without access to details about who you are unless you share them with me directly.\n\nIf you'd like me to refer to you by a specific name, feel free to tell me what you'd like to be called!"

### Wait, wha??

We just told you!

What's going on??

Here's the thing: every call to an LLM is completely STATELESS. It's a totally new call, every single time. As AI engineers, it's OUR JOB to devise techniques to give the impression that the LLM has a "memory".

In [24]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm Navi!"},
    {"role": "assistant", "content": "Hi Navi! 👋 Welcome! Nice to meet you. I'm MiniMax-M3, your AI assistant. \n\nHow can I help you today? Whether you have questions, need assistance with something, or just want to chat, I'm here for you! 😊"},
    {"role": "user", "content": "What's my name?"}
    ]

In [25]:
response = openai.chat.completions.create(model=MODEL, messages=messages)
response.choices[0].message.content

"Your name is **Navi**! You introduced yourself at the start of our conversation. 😊\n\nIs there anything else you'd like to know or talk about?"

## To recap

With apologies if this is obvious to you - but it's still good to reinforce:

1. Every call to an LLM is stateless
2. We pass in the entire conversation so far in the input prompt, every time
3. This gives the illusion that the LLM has memory - it apparently keeps the context of the conversation
4. But this is a trick; it's a by-product of providing the entire conversation, every time
5. An LLM just predicts the most likely next tokens in the sequence; if that sequence contains "My name is Ed" and later "What's my name?" then it will predict.. Ed!

The ChatGPT product uses exactly this trick - every time you send a message, it's the entire conversation that gets passed in.

"Does that mean we have to pay extra each time for all the conversation so far"

For sure it does. And that's what we WANT. We want the LLM to predict the next tokens in the sequence, looking back on the entire conversation. We want that compute to happen, so we need to pay the electricity bill for it!

